<h1><center>Computational Quantum Physics - PHYS 463</center></h1>

<p><center> <b>Lecturer:</b> <i>Prof. G. Carleo</i> </center><p>
    
<p><center> <b>Assistants: </b> <i>gabriel.pescia@epfl.ch, alessandro.sinibaldi@epfl.ch, clemens.giuliani@epfl.ch, linda.mauron@epfl.ch </i> </center><p>

# Exercise 8 - Template

In [ ]:
# Run this if you are on Windows and you don't want to use GPU
import os

os.environ["JAX_PLATFORM_NAME"] = "cpu"

In [ ]:
from typing import Any, Tuple
from flax import nnx
import flax.linen as nn
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import netket as nk
import numpy as np

/home/francesco/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Setup

# Number of spins
N = 20
# Number of samples
Ns = 1024
# How many Markov chains to run in parallel
n_chains = 16
# How many samples to discard
n_discard = 128

# Graph
g = nk.graph.Chain(N, pbc=True)
# Hilbert space
hi = nk.hilbert.Spin(s=1 / 2, N=g.n_nodes)
# Hamiltonian
ha = nk.operator.Ising(hilbert=hi, graph=g, h=1)
# Sampler (single-spin flips)
sa = nk.sampler.MetropolisLocal(hi, n_chains=n_chains)
# Optimizer
op = nk.optimizer.Sgd(0.01)

In [ ]:
# Exact diagonalization (lanczos_ed returns an array of eigenvalues)
E0 = nk.exact.lanczos_ed(ha)[0]
print("Exact ground-state energy:", E0)

In [ ]:
class FFNN(nn.Module):
    # Number of hidden neurons per hidden layer
    width: int = 5
    # Number of hidden layers
    n_layers: int = 1

    @nn.compact
    def __call__(self, x):
        # x has shape (n_samples, N)
        # Hidden layers: x^(l) = ReLU(W^(l-1) x^(l-1) + b^(l))
        for _ in range(self.n_layers):
            x = nn.relu(nn.Dense(self.width)(x))
        # Output layer with a single unit: log psi(s) = ReLU(W x + b)
        x = nn.relu(nn.Dense(1)(x))
        # (n_samples, 1) -> (n_samples,)
        return x.squeeze(-1)

In [ ]:
ma = FFNN(width=N // 4, n_layers=1)

## Problem 8.1 b) VMC with the MLP Ansatz, N = 20

In [ ]:
# Variational state object
vs = nk.vqs.MCState(sa, ma, n_samples=Ns, n_discard_per_chain=n_discard)
# VMC optimization driver
vmc = nk.VMC(ha, op, variational_state=vs)

In [ ]:
# Log the energy at every iteration to plot the convergence
log = nk.logging.RuntimeLog()
vmc.run(n_iter=1000, out=log)

In [ ]:
ffnn_energy = vs.expect(ha)
error = abs((ffnn_energy.mean - E0) / E0)
print("Optimized energy and relative error: ", ffnn_energy, error)

Optimized energy and relative error:  -19.68 ± 0.14 [σ²=20.04, R̂=1.0084] [0.22782152]


In [ ]:
data = log.data["Energy"]
iters = np.asarray(data.iters)
E_mean = np.asarray(data.Mean.real)
E_sigma = np.asarray(data.Sigma)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.errorbar(iters, E_mean, yerr=E_sigma, lw=1, label=f"FFNN (width={ma.width}, layers={ma.n_layers})")
ax1.axhline(E0, color="k", ls="--", label="Exact")
ax1.set_xlabel("Iteration")
ax1.set_ylabel("Energy")
ax1.legend()

ax2.semilogy(iters, np.abs((E_mean - E0) / E0))
ax2.set_xlabel("Iteration")
ax2.set_ylabel(r"$|E - E_0| / |E_0|$")

fig.tight_layout()
plt.show()

## Problem 8.1 c) Dependence on the width and depth of the network

In [ ]:
def run_ffnn(width, n_layers, n_iter=1000, seed=0):
    """Optimize an FFNN Ansatz with VMC and return its final energy and training log."""
    model = FFNN(width=width, n_layers=n_layers)
    # Fixed seeds so that only the architecture changes between runs
    vstate = nk.vqs.MCState(
        sa, model, n_samples=Ns, n_discard_per_chain=n_discard, seed=seed, sampler_seed=seed
    )
    driver = nk.VMC(ha, nk.optimizer.Sgd(0.01), variational_state=vstate)
    run_log = nk.logging.RuntimeLog()
    driver.run(n_iter=n_iter, out=run_log, show_progress=False)

    E = vstate.expect(ha)
    E_mean = float(E.mean.real)
    return {
        "width": width,
        "n_layers": n_layers,
        "n_params": vstate.n_parameters,
        "E": E_mean,
        "E_err": float(E.error_of_mean),
        "var": float(E.variance),
        "rel_err": abs((E_mean - E0) / E0),
        "log": run_log.data["Energy"],
    }

In [ ]:
# Widths N/4, N/2, N, 2N and 1 to 3 hidden layers (this takes a while on CPU).
# With plain SGD the outcome depends strongly on the random initialization
# (some runs get stuck in a local minimum), so every architecture is run with several seeds.
widths = [N // 4, N // 2, N, 2 * N]
depths = [1, 2, 3]
seeds = [0, 1, 2]

results = []
for n_layers in depths:
    for width in widths:
        for seed in seeds:
            r = run_ffnn(width, n_layers, seed=seed)
            r["seed"] = seed
            results.append(r)
            print(
                f"layers={n_layers} width={width:3d} seed={seed} params={r['n_params']:5d}  "
                f"E={r['E']:.4f} ± {r['E_err']:.4f}  var={r['var']:.3f}  rel_err={r['rel_err']:.2e}"
            )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for i, n_layers in enumerate(depths):
    color = f"C{i}"
    label = f"{n_layers} hidden layer" + ("s" if n_layers > 1 else "")
    rs = [r for r in results if r["n_layers"] == n_layers]
    # Best run over seeds (solid line) and every individual run (faint points)
    best = [min((r for r in rs if r["width"] == w), key=lambda r: r["rel_err"]) for w in widths]
    axes[0].semilogy(widths, [b["rel_err"] for b in best], "o-", color=color, label=label)
    axes[0].semilogy([r["width"] for r in rs], [r["rel_err"] for r in rs], "x", color=color, alpha=0.4)
    axes[1].loglog([b["n_params"] for b in best], [b["rel_err"] for b in best], "o-", color=color, label=label)

axes[0].set_xlabel("Width (hidden neurons per layer)")
axes[0].set_xticks(widths)
axes[0].set_title("Lines: best of seeds, crosses: all runs")
axes[1].set_xlabel("Number of variational parameters")
axes[1].set_title("Best of seeds")
for ax in axes[:2]:
    ax.set_ylabel(r"$|E - E_0| / |E_0|$")
    ax.legend()

# Convergence of all one-hidden-layer runs of width N/4: same Ansatz, different seeds
for r in results:
    if r["n_layers"] == 1 and r["width"] == widths[0]:
        axes[2].plot(r["log"].iters, r["log"].Mean.real, lw=1, label=f"seed={r['seed']}")
axes[2].axhline(E0, color="k", ls="--", label="Exact")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Energy")
axes[2].set_title(f"1 hidden layer, width {widths[0]}")
axes[2].legend()

fig.tight_layout()
plt.show()

### Discussion

Exact energy: $E_0 = -25.491$. Relative error after 1000 SGD iterations (learning rate 0.01), for seeds 0 / 1 / 2:

| hidden layers | width 5 | width 10 | width 20 | width 40 |
|---|---|---|---|---|
| 1 | 1.4 / 6.8 / 7.2 % | 7.6 / 1.5 / 6.3 % | 1.3 / 1.3 / 5.8 % | 1.5 / 1.5 / 1.3 % |
| 2 | 1.6 / 6.9 / 5.8 % | 8.0 / 1.5 / 5.5 % | 21 / 1.7 / 1.3 % | 21 / 1.2 / 1.3 % |
| 3 | 1.6 / 8.0 / 1.5 % | 1.4 / 5.2 / 5.6 % | 1.1 / 1.5 / 21 % | 1.1 / 22 / 1.1 % |

**The runs end in one of three places, and which one depends mostly on the random initialization, not on the architecture:**

1. $E \approx -25.1$ to $-25.2$ (relative error 1.1–1.7 %, variance ≈ 1.1): the optimization worked. 20 of the 36 runs.
2. $E \approx -23.5$ to $-24.2$ (relative error 5–8 %, variance ≈ 3): a plateau. The right panel shows that a run can sit there for a long time and then escape (seed 0 leaves it around iteration 300). With plain SGD and 1000 iterations, 12 runs never escaped.
3. $E \approx -20$ (relative error ≈ 21 %, variance ≈ 20): these runs never move away from their starting energy. $-20 = -\Gamma N$ is the energy of a constant wave function, for which $\langle\sigma^z_i\sigma^z_{i+1}\rangle = 0$. This means the output ReLU is zero for (nearly) all sampled configurations: $\log\psi = 0$, and its gradient vanishes, so SGD cannot move. This only happened for the deep and wide networks (≥ 2 layers, width ≥ 20).

**Width.** For one hidden layer, the best energy improves only slightly with width, from 1.4 % at width $N/4$ to 1.3 % at width $2N$. Wider networks do make the optimization more reliable: at width 40, all three seeds converge. A single hidden layer of width 5 (111 parameters) can already reach ≈ 1.4 %, so the error is not limited by the size of the network but by the optimization (plain SGD, fixed learning rate, 1000 iterations) and by Monte Carlo noise.

**Depth.** Deeper networks reach slightly lower energies when they converge. The best runs go from 1.3 % (1 layer) to 1.2 % (2 layers) to 1.1 % (3 layers), all at width 40. The gain is small compared to the spread between seeds, and deep networks fail more often: they add the dead-output failure mode on top of the plateau. More parameters make the Ansatz more expressive but harder to train with plain SGD. Stochastic reconfiguration (`preconditioner=nk.optimizer.SR(...)`) or a different output layer (e.g. summing the last hidden layer instead of a single ReLU unit) would likely remove most of these failures.

## Problem 8.2 (Optional) Convolutional Neural Network Ansatz

Each hidden layer is a 1D convolution with circular padding, which respects the periodic boundary conditions, followed by a ReLU.
The feature maps are then pooled over the lattice sites and summed over the features:

$$\log\psi(s) = \sum_{f} \operatorname{pool}_{i}\, x^{(L)}_{i,f}, \qquad \operatorname{pool} \in \{\text{sum}, \text{mean}, \text{max}\}.$$

Because the same filter is applied at every site and the pooling runs over all sites, $\log\psi(Ts) = \log\psi(s)$ for every translation $T$ of the chain.

In [ ]:
class CNN(nn.Module):
    # Number of filters (output channels) per convolutional layer
    features: int = 4
    # Size of the convolution filter (number of neighbouring sites it sees)
    kernel_size: int = 3
    # Number of convolutional layers
    n_layers: int = 1
    # Pooling over the lattice sites: jnp.sum, jnp.mean or jnp.max
    pooling: Any = jnp.sum

    @nn.compact
    def __call__(self, x):
        # (n_samples, N) -> (n_samples, N, 1): one input channel per site
        x = x.reshape(x.shape[0], -1, 1)
        for _ in range(self.n_layers):
            # Circular padding implements the periodic boundary conditions
            x = nn.Conv(self.features, kernel_size=(self.kernel_size,), padding="CIRCULAR")(x)
            x = nn.relu(x)
        # Pool over the sites, then sum over the features: (n_samples, N, features) -> (n_samples,)
        return jnp.sum(self.pooling(x, axis=1), axis=-1)


# Check the output shape and the translation invariance
cnn = CNN()
x_test = hi.random_state(jax.random.PRNGKey(0), 5)
p_test = cnn.init(jax.random.PRNGKey(0), x_test)
print(cnn.apply(p_test, x_test).shape)
print(jnp.allclose(cnn.apply(p_test, x_test), cnn.apply(p_test, jnp.roll(x_test, 3, axis=1)), atol=1e-5))

In [ ]:
def run_cnn(kernel_size, pooling, features=4, n_iter=1000, seed=0):
    """Optimize a CNN Ansatz with VMC (same settings as the MLP) and return its final energy and log."""
    model = CNN(features=features, kernel_size=kernel_size, pooling=pooling)
    vstate = nk.vqs.MCState(
        sa, model, n_samples=Ns, n_discard_per_chain=n_discard, seed=seed, sampler_seed=seed
    )
    driver = nk.VMC(ha, nk.optimizer.Sgd(0.01), variational_state=vstate)
    run_log = nk.logging.RuntimeLog()
    driver.run(n_iter=n_iter, out=run_log, show_progress=False)

    E = vstate.expect(ha)
    E_mean = float(E.mean.real)
    return {
        "kernel_size": kernel_size,
        "pooling": pooling.__name__,
        "n_params": vstate.n_parameters,
        "E": E_mean,
        "E_err": float(E.error_of_mean),
        "var": float(E.variance),
        "rel_err": abs((E_mean - E0) / E0),
        "log": run_log.data["Energy"],
    }


# Filter sizes from nearest neighbours up to (almost) the whole chain, and three pooling functions.
# One CNN run takes a few minutes on CPU.
kernel_sizes = [3, 5, 9, 15]
poolings = [jnp.sum, jnp.mean, jnp.max]

cnn_results = []
for pooling in poolings:
    for k in kernel_sizes:
        r = run_cnn(k, pooling)
        cnn_results.append(r)
        print(
            f"pool={r['pooling']:4s} kernel={k:2d} params={r['n_params']:3d}  "
            f"E={r['E']:.4f} ± {r['E_err']:.4f}  var={r['var']:.3f}  rel_err={r['rel_err']:.2e}"
        )

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for pool_name in ["sum", "mean", "max"]:
    rs = [r for r in cnn_results if r["pooling"] == pool_name]
    axes[0].semilogy([r["kernel_size"] for r in rs], [r["rel_err"] for r in rs], "o-", label=f"{pool_name} pooling")
axes[0].set_xlabel("Filter size")
axes[0].set_xticks(kernel_sizes)
axes[0].set_ylabel(r"$|E - E_0| / |E_0|$")
axes[0].set_title("CNN")
axes[0].legend()

# CNN vs MLP (all runs of 8.1 c) as a function of the number of parameters
axes[1].loglog([r["n_params"] for r in results], [r["rel_err"] for r in results], "x", color="gray", label="MLP (all runs)")
axes[1].loglog([r["n_params"] for r in cnn_results], [r["rel_err"] for r in cnn_results], "o", color="C3", label="CNN (all runs)")
axes[1].set_xlabel("Number of variational parameters")
axes[1].set_ylabel(r"$|E - E_0| / |E_0|$")
axes[1].set_title("CNN vs MLP")
axes[1].legend()

# Convergence for the three pooling functions at filter size 5
for r in cnn_results:
    if r["kernel_size"] == 5:
        axes[2].plot(r["log"].iters, r["log"].Mean.real, lw=1, label=f"{r['pooling']} pooling")
axes[2].axhline(E0, color="k", ls="--", label="Exact")
axes[2].set_xlabel("Iteration")
axes[2].set_ylabel("Energy")
axes[2].set_title("CNN, filter size 5")
axes[2].legend()

fig.tight_layout()
plt.show()